# TAREA 2.2: Adquisición de Datos Climáticos (AEMET OpenData)

## 1. Introducción y Objetivos

En esta fase del proyecto, el objetivo principal es enriquecer nuestro dataset histórico de incendios forestales de la Comunidad Valenciana con **datos meteorológicos reales** del día del incendio y los días previos. Las condiciones climáticas (temperatura, viento, humedad, precipitación) son factores críticos tanto para la ignición como para la propagación de un incendio forestal.

### Retos Técnicos Abordados:

1. **Volumen y Límites de API**: La API de AEMET OpenData impone límites estrictos de peticiones por minuto y restringe las consultas de datos diarios a ventanas máximas de 6 meses. Extraer datos para 13 años (2010-2022) requirió el diseño de un pipeline robusto con gestión de reintentos, pausas automáticas (rate-limiting) y almacenamiento en caché local para evitar la pérdida de progreso en caso de fallo de red.
2. **Asignación Espacial Eficiente**: En lugar de utilizar la fórmula de Haversine iterando fila por fila (costoso computacionalmente), se ha implementado un enfoque vectorizado utilizando `BallTree` de Scikit-Learn. Esto permite encontrar la estación meteorológica más cercana a cada uno de los 4.477 incendios de forma casi instantánea.
3. **Ingeniería de Variables Temporales**: No solo nos interesa el clima del día del incendio, sino el contexto previo. El pipeline precalcula variables vitales para el riesgo de incendio como la **precipitación acumulada de los 7 días anteriores** o los **días consecutivos sin lluvia** previos a la ignición.

Toda la lógica compleja ha sido encapsulada en el módulo `src/descarga_aemet.py`, manteniendo este notebook limpio para la ejecución y validación.

## 2. Configuración y Carga del Módulo

In [1]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


sys.path.insert(0, os.path.abspath('../src'))

from descarga_aemet import (
    cargar_api_key, 
    obtener_estaciones_cv, 
    asignar_estacion_cercana, 
    ejecutar_pipeline
)


pd.set_option('display.max_columns', 50)

## 3. Validación Inicial: Estaciones en la C. Valenciana

Antes de lanzar la descarga masiva, verificamos que podemos conectar con AEMET y que estamos identificando correctamente las estaciones dentro de nuestro ámbito geográfico.

In [ ]:
# Cargar API key
api_key = cargar_api_key()
print(" API Key cargada correctamente.")

# Descargar el inventario de estaciones y filtrar por la C. Valenciana
df_estaciones = obtener_estaciones_cv(api_key)

display(df_estaciones[['indicativo', 'nombre', 'provincia', 'lat_est', 'lon_est']].head(10))

## 4. Ejecución del Pipeline Completo

El siguiente bloque ejecuta el proceso de descarga cruzada. Si se ejecuta por primera vez, el proceso **tomará aproximadamente 45-50 minutos**. 

Si el proceso se interrumpe, el sistema de caché implementado en el script (basado en archivos JSON por estación en `datos/external/cache_aemet/`) permitirá reanudar la descarga exactamente donde se quedó.

In [ ]:
print("Iniciando pipeline de descarga y cruce AEMET...")



## 5. Auditoría del Dataset Enriquecido

Una vez ha finalizado el pipeline, cargamos el dataset resultante para comprobar su calidad.

In [ ]:

ruta_final = os.path.join('..', 'datos', 'processed', '02_incendios_clima_CV.csv')

if os.path.exists(ruta_final):
    df_final = pd.read_csv(ruta_final, sep=';')
    print(f" Dataset cargado con éxito: {len(df_final)} registros y {len(df_final.columns)} columnas.")
    
    print("\n--- MUESTRA DEL DATASET (Columnas Climáticas) ---")
    columnas_clima = ['Municipio', 'fecha_ini', 'nombre_estacion', 'distancia_estacion_km', 
                      'temp_max', 'precipitacion', 'viento_medio', 'dias_sin_lluvia']
    
    
    display(df_final[df_final['temp_max'].notna()][columnas_clima].head())
else:
    print(" El archivo final no existe aún. Debes ejecutar el pipeline primero.")

### 5.1 Calidad del Cruce y Valores Nulos

En datos meteorológicos históricos es común tener lagunas (estaciones que dejaron de funcionar, mantenimientos, etc.). Vamos a evaluar cuántos incendios se han quedado sin datos climáticos.

In [ ]:
if 'df_final' in locals():
    variables_climaticas = ['temp_media', 'temp_max', 'temp_min', 'precipitacion', 
                            'humedad_media', 'viento_medio', 'racha_max', 
                            'prec_acum_7d', 'dias_sin_lluvia', 'tmax_max_7d']
    
    completitud = []
    
    for col in variables_climaticas:
        if col in df_final.columns:
            nulos = df_final[col].isna().sum()
            porcentaje_completitud = (1 - (nulos / len(df_final))) * 100
            completitud.append({'Variable': col, 'Completitud (%)': round(porcentaje_completitud, 2)})
            
    df_completitud = pd.DataFrame(completitud).sort_values('Completitud (%)', ascending=False)
    
    plt.figure(figsize=(10, 6))
    bars = plt.barh(df_completitud['Variable'], df_completitud['Completitud (%)'], color='skyblue')
    plt.axvline(x=80, color='r', linestyle='--', label='Umbral 80%')
    plt.xlabel('Porcentaje de Completitud (%)')
    plt.title('Disponibilidad de Variables Climáticas en Incendios')
    plt.gca().invert_yaxis()
    plt.legend()
    
    
    for bar in bars:
        width = bar.get_width()
        plt.text(width + 1, bar.get_y() + bar.get_height()/2, f'{width}%', 
                 ha='left', va='center', fontweight='bold')
        
    plt.tight_layout()
    plt.show()